# 21_E13 - Pipeline común de inferencia multiplanar

Este notebook arma el primer pipeline común para usar los dos modelos finales del proyecto:

- **Sagital SPIDER**: checkpoint consolidado en E12.
- **Axial T2 Al-Kafri/Sudirman**: checkpoint final limpio E10.

El objetivo no es entrenar, sino estandarizar inferencia, overlays, métricas rápidas y *quality flags* para avanzar hacia el agente IA / MVP.

Salidas principales:

- `results/E13_multiplanar_inference_pipeline/E13_multiplanar_pipeline_report.json`
- `docs/E13_multiplanar_inference_pipeline_conclusion.md`
- figuras de ejemplo sagital y axial en `figures/`


In [ ]:
# Instalación segura de dependencias en Colab
try:
    import SimpleITK  # noqa
    import pydicom  # noqa
    import skimage  # noqa
    import scipy  # noqa
except Exception:
    %pip install -q SimpleITK pydicom scikit-image scipy


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import SimpleITK as sitk
import pydicom
from skimage.transform import resize
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")

SPIDER_ROOT = PFI_ROOT / "data" / "SPIDER"
E9_ROOT = PFI_ROOT / "results" / "E9_alkafri_axial_t2_final_labels_baseline"
E5_MULTICLASS_ROOT = PFI_ROOT / "results" / "E5_multiclase_agrupado"
E5_HOLDOUT_ROOT = PFI_ROOT / "results" / "E5_multiclase_holdout"

E13_ROOT = PFI_ROOT / "results" / "E13_multiplanar_inference_pipeline"
FIGURES_ROOT = PFI_ROOT / "figures"
DOCS_ROOT = PFI_ROOT / "docs"
MODELS_ROOT = PFI_ROOT / "models"

for p in [E13_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SAGITTAL_MODEL_PATH = MODELS_ROOT / "E12_sagittal_multiclass_final_best.pt"
if not SAGITTAL_MODEL_PATH.exists():
    SAGITTAL_MODEL_PATH = E5_MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_best.pt"

AXIAL_MODEL_PATH = MODELS_ROOT / "E10_axial_t2_final_training_clean_best.pt"

AXIAL_SPLIT_PATH = E9_ROOT / "E9_t2_final_labels_curated_split.csv"
SAGITTAL_HOLDOUT_CASES = E5_HOLDOUT_ROOT / "E5_multiclass_holdout_selected_cases.csv"

required = {
    "SAGITTAL_MODEL_PATH": SAGITTAL_MODEL_PATH,
    "AXIAL_MODEL_PATH": AXIAL_MODEL_PATH,
    "AXIAL_SPLIT_PATH": AXIAL_SPLIT_PATH,
}

for name, path in required.items():
    print(name, "->", path, "| exists:", path.exists())
    assert path.exists(), f"Falta {name}: {path}"

print("SAGITTAL_HOLDOUT_CASES:", SAGITTAL_HOLDOUT_CASES, SAGITTAL_HOLDOUT_CASES.exists())


## 1. Arquitecturas compatibles con los checkpoints finales

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SagittalUNet2DMulticlass(nn.Module):
    def __init__(self, in_channels=1, num_classes=4, base_channels=16):
        super().__init__()
        b = base_channels
        self.enc1 = DoubleConv(in_channels, b)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(b, b * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(b * 2, b * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(b * 4, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(b * 2, b)
        self.out_conv = nn.Conv2d(b, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        m = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(m), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


class AxialDC(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(a, b, 3, padding=1),
            nn.BatchNorm2d(b),
            nn.ReLU(True),
            nn.Conv2d(b, b, 3, padding=1),
            nn.BatchNorm2d(b),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.net(x)


class AxialUNet(nn.Module):
    def __init__(self, num_classes=6, base=16):
        super().__init__()
        b = base
        self.e1 = AxialDC(1, b)
        self.e2 = AxialDC(b, b * 2)
        self.e3 = AxialDC(b * 2, b * 4)
        self.pool = nn.MaxPool2d(2)
        self.mid = AxialDC(b * 4, b * 8)
        self.u3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2)
        self.d3 = AxialDC(b * 8, b * 4)
        self.u2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2)
        self.d2 = AxialDC(b * 4, b * 2)
        self.u1 = nn.ConvTranspose2d(b * 2, b, 2, 2)
        self.d1 = AxialDC(b * 2, b)
        self.out = nn.Conv2d(b, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        m = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(m), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


## 2. Carga de modelos finales

In [ ]:
IMG_SIZE = 256

# Sagital
sag_ckpt = torch.load(SAGITTAL_MODEL_PATH, map_location=DEVICE)
sag_num_classes = int(sag_ckpt.get("num_classes", 4))
sag_base = int(sag_ckpt.get("base_channels", 16))
sag_target_size = tuple(sag_ckpt.get("target_size", (256, 256)))
sag_axis = int(sag_ckpt.get("sagittal_axis", 2))
sag_slice_strategy = str(sag_ckpt.get("slice_strategy", "center_window_best_prediction"))
sag_label_mapping = {int(k): int(v) for k, v in sag_ckpt.get("label_group_mapping", {}).items()}

sag_model = SagittalUNet2DMulticlass(
    in_channels=1,
    num_classes=sag_num_classes,
    base_channels=sag_base,
).to(DEVICE)
sag_model.load_state_dict(sag_ckpt["model_state_dict"])
sag_model.eval()

# Axial
axial_state = torch.load(AXIAL_MODEL_PATH, map_location=DEVICE)
if isinstance(axial_state, dict) and "model_state_dict" in axial_state:
    axial_state = axial_state["model_state_dict"]

ax_model = AxialUNet(num_classes=6, base=16).to(DEVICE)
ax_model.load_state_dict(axial_state)
ax_model.eval()

model_registry = {
    "sagittal_spider": {
        "model": sag_model,
        "path": str(SAGITTAL_MODEL_PATH),
        "num_classes": sag_num_classes,
        "target_size": sag_target_size,
        "axis": sag_axis,
        "slice_strategy": sag_slice_strategy,
        "class_names": {0: "background", 1: "vertebra_group", 2: "canal", 3: "disc_group"},
    },
    "axial_t2_alkafri": {
        "model": ax_model,
        "path": str(AXIAL_MODEL_PATH),
        "num_classes": 6,
        "target_size": (256, 256),
        "class_names": {
            0: "background_250",
            1: "raw_0",
            2: "raw_50",
            3: "raw_100",
            4: "raw_150",
            5: "raw_200",
        },
    },
}

print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in model_registry.items()}, indent=2, ensure_ascii=False))


## 3. Funciones comunes de preprocesamiento, inferencia y calidad

In [ ]:
def norm_img(x):
    x = x.astype(np.float32)
    p1, p99 = np.percentile(x, [1, 99])
    if p99 <= p1:
        return np.zeros_like(x, dtype=np.float32)
    return ((np.clip(x, p1, p99) - p1) / (p99 - p1 + 1e-8)).astype(np.float32)


def resize_img(x, size=256):
    return resize(x, (size, size), order=1, preserve_range=True, anti_aliasing=True).astype(np.float32)


def resize_lbl(y, size=256):
    return resize(y.astype(np.float32), (size, size), order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)


@torch.no_grad()
def predict_2d(model, image_2d, device=DEVICE):
    x = torch.from_numpy(image_2d[None, None]).float().to(device)
    logits = model(x)
    prob = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    pred = prob.argmax(axis=0).astype(np.uint8)
    confidence = prob.max(axis=0).astype(np.float32)
    return pred, prob, confidence


def dice_by_class(pred, gt, num_classes, class_names=None):
    rows = []
    for c in range(num_classes):
        p = pred == c
        g = gt == c
        inter = int(np.logical_and(p, g).sum())
        denom = int(p.sum() + g.sum())
        union = int(np.logical_or(p, g).sum())
        dice = (2 * inter / denom) if denom > 0 else np.nan
        iou = (inter / union) if union > 0 else np.nan
        rows.append({
            "class_id": c,
            "class_name": class_names.get(c, str(c)) if class_names else str(c),
            "dice": dice,
            "iou": iou,
            "pred_pixels": int(p.sum()),
            "gt_pixels": int(g.sum()),
        })
    return pd.DataFrame(rows)


def quality_flags(mask, confidence=None, min_fg=0.002, max_fg=0.35):
    fg = mask > 0
    fg_ratio = float(fg.mean())
    labels, n_components = ndi.label(fg.astype(np.uint8))
    present_classes = sorted([int(v) for v in np.unique(mask) if int(v) != 0])
    flags = []

    if fg_ratio < min_fg:
        flags.append("foreground_muy_bajo")
    if fg_ratio > max_fg:
        flags.append("foreground_muy_alto")
    if len(present_classes) == 0:
        flags.append("sin_clases_no_fondo")
    if n_components > 10:
        flags.append("muchos_componentes")

    out = {
        "foreground_ratio": fg_ratio,
        "n_components": int(n_components),
        "present_classes": present_classes,
        "flags": flags,
    }

    if confidence is not None:
        out["mean_confidence"] = float(np.mean(confidence))
        out["mean_fg_confidence"] = float(np.mean(confidence[fg])) if fg.any() else None

    return out


def save_overlay_figure(image, pred, out_path, title="", gt=None):
    ncols = 4 if gt is not None else 3
    fig, ax = plt.subplots(1, ncols, figsize=(4 * ncols, 4))

    ax[0].imshow(image, cmap="gray")
    ax[0].set_title("Imagen")

    if gt is not None:
        ax[1].imshow(gt, cmap="nipy_spectral")
        ax[1].set_title("GT")
        ax[2].imshow(pred, cmap="nipy_spectral")
        ax[2].set_title("Pred")
        ax[3].imshow(image, cmap="gray")
        ax[3].imshow(np.ma.masked_where(pred <= 0, pred), cmap="autumn", alpha=0.45)
        ax[3].set_title("Overlay pred")
    else:
        ax[1].imshow(pred, cmap="nipy_spectral")
        ax[1].set_title("Pred")
        ax[2].imshow(image, cmap="gray")
        ax[2].imshow(np.ma.masked_where(pred <= 0, pred), cmap="autumn", alpha=0.45)
        ax[2].set_title("Overlay pred")

    for a in ax:
        a.axis("off")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    return out_path


## 4. Pipeline axial T2

In [ ]:
AXIAL_RAW_VALUES = np.array([0, 50, 100, 150, 200, 250], dtype=np.int16)
AXIAL_RAW_TO_CLASS = {250: 0, 0: 1, 50: 2, 100: 3, 150: 4, 200: 5}
AXIAL_CLASS_NAMES = model_registry["axial_t2_alkafri"]["class_names"]


def read_axial_dicom(path):
    ds = pydicom.dcmread(str(path), force=True)
    return ds.pixel_array.astype(np.float32)


def read_axial_label(path):
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.int16)
    flat = arr.reshape(-1)
    snapped = AXIAL_RAW_VALUES[
        np.abs(flat[:, None] - AXIAL_RAW_VALUES[None, :]).argmin(axis=1)
    ].reshape(arr.shape)

    y = np.zeros_like(snapped, dtype=np.uint8)
    for rv, c in AXIAL_RAW_TO_CLASS.items():
        y[snapped == rv] = c
    return y


def infer_axial_t2(image_file_path):
    img_raw = read_axial_dicom(image_file_path)
    img = resize_img(norm_img(img_raw), IMG_SIZE)
    pred, prob, conf = predict_2d(ax_model, img)
    q = quality_flags(pred, conf, min_fg=0.01, max_fg=0.25)
    return {
        "plane": "axial",
        "model_key": "axial_t2_alkafri",
        "image": img,
        "pred": pred,
        "prob": prob,
        "confidence": conf,
        "quality": q,
    }


axial_df = pd.read_csv(AXIAL_SPLIT_PATH)
axial_df = axial_df[axial_df["modality"].astype(str).eq("T2")].copy()
axial_test_df = axial_df[axial_df["split"].eq("test")].copy()

print("Axial T2 test rows:", len(axial_test_df))
display(axial_test_df.head(3))


In [ ]:
axial_rows = []
axial_metric_rows = []

for i, (_, r) in enumerate(axial_test_df.head(6).iterrows(), 1):
    result = infer_axial_t2(r["image_file_path"])
    gt = resize_lbl(read_axial_label(r["final_label_file_path"]), IMG_SIZE)

    metrics = dice_by_class(
        result["pred"],
        gt,
        num_classes=6,
        class_names=AXIAL_CLASS_NAMES,
    )
    metrics["example_id"] = i
    metrics["plane"] = "axial"
    metrics["case_id_norm"] = str(r.get("case_id_norm", ""))
    metrics["disc_id"] = int(r.get("disc_id", -1))
    axial_metric_rows.append(metrics)

    out_path = FIGURES_ROOT / f"E13_axial_t2_example_{i:02d}.png"
    save_overlay_figure(
        result["image"],
        result["pred"],
        out_path,
        title=f"E13 axial example {i} | case={r.get('case_id_norm')} | D={r.get('disc_id')}",
        gt=gt,
    )

    axial_rows.append({
        "example_id": i,
        "plane": "axial",
        "case_id_norm": str(r.get("case_id_norm", "")),
        "disc_id": int(r.get("disc_id", -1)),
        "figure_path": str(out_path),
        **result["quality"],
    })

axial_examples_df = pd.DataFrame(axial_rows)
axial_metrics_df = pd.concat(axial_metric_rows, ignore_index=True) if axial_metric_rows else pd.DataFrame()

axial_examples_df.to_csv(E13_ROOT / "E13_axial_examples_quality.csv", index=False)
axial_metrics_df.to_csv(E13_ROOT / "E13_axial_examples_metrics_by_class.csv", index=False)

display(axial_examples_df)
display(axial_metrics_df.head(20))


## 5. Pipeline sagital SPIDER

In [ ]:
SAGITTAL_CLASS_NAMES = model_registry["sagittal_spider"]["class_names"]


def read_mha_xyz(path):
    img = sitk.ReadImage(str(path))
    arr_zyx = sitk.GetArrayFromImage(img).astype(np.float32)
    arr_xyz = np.transpose(arr_zyx, (2, 1, 0))
    return arr_xyz


def extract_slice(volume, axis, idx):
    if axis == 0:
        return volume[idx, :, :]
    if axis == 1:
        return volume[:, idx, :]
    if axis == 2:
        return volume[:, :, idx]
    raise ValueError("axis debe ser 0, 1 o 2")


def group_sagittal_mask(mask_2d):
    if not sag_label_mapping:
        out = np.zeros_like(mask_2d, dtype=np.uint8)
        out[np.isin(mask_2d, list(range(1, 100)))] = 1
        out[mask_2d == 100] = 2
        out[np.isin(mask_2d, list(range(201, 300)))] = 3
        return out

    out = np.zeros_like(mask_2d, dtype=np.uint8)
    for original_value, grouped_class in sag_label_mapping.items():
        out[mask_2d == int(original_value)] = int(grouped_class)
    return out


def select_sagittal_slice(volume, axis=2, radius=3):
    n = volume.shape[axis]
    center = n // 2
    candidates = list(range(max(0, center - radius), min(n, center + radius + 1)))

    best = None
    for idx in candidates:
        sl = extract_slice(volume, axis, idx)
        img = resize_img(norm_img(sl), IMG_SIZE)
        pred, prob, conf = predict_2d(sag_model, img)
        score = float(prob[1:].sum(axis=0).mean())
        if best is None or score > best["score"]:
            best = {"idx": idx, "image": img, "pred": pred, "prob": prob, "confidence": conf, "score": score}

    return best


def candidate_case_id_from_row(row):
    if "case_id" in row and pd.notna(row["case_id"]):
        return str(row["case_id"])
    for column in ["file_name", "image_path", "source_image_path", "mask_path", "source_mask_path"]:
        if column in row and pd.notna(row[column]):
            return Path(str(row[column])).stem
    return None


def resolve_spider_path(row, kind):
    columns = (
        ["image_path", "source_image_path", "image", "img_path"]
        if kind == "image"
        else ["mask_path", "source_mask_path", "mask", "seg_path"]
    )

    direct = []
    for col in columns:
        if col in row and pd.notna(row[col]):
            direct.append(Path(str(row[col])))

    case_id = candidate_case_id_from_row(row)
    base = []
    if case_id:
        if kind == "image":
            base += [
                SPIDER_ROOT / "images" / "images" / f"{case_id}.mha",
                SPIDER_ROOT / "images" / f"{case_id}.mha",
                SPIDER_ROOT / f"{case_id}.mha",
            ]
        else:
            base += [
                SPIDER_ROOT / "masks" / "masks" / f"{case_id}.mha",
                SPIDER_ROOT / "masks" / f"{case_id}.mha",
                SPIDER_ROOT / f"{case_id}.mha",
            ]

    for p in direct + base:
        if p.exists():
            return p

    if case_id and SPIDER_ROOT.exists():
        matches = list(SPIDER_ROOT.rglob(f"{case_id}.mha"))
        if kind == "image":
            matches = [m for m in matches if "mask" not in str(m).lower()]
        else:
            matches = [m for m in matches if "mask" in str(m).lower() or "seg" in str(m).lower()]
        if matches:
            return matches[0]

    return None


def infer_sagittal_spider(image_file_path, mask_file_path=None):
    volume = read_mha_xyz(image_file_path)
    selected = select_sagittal_slice(volume, axis=sag_axis, radius=3)

    gt = None
    if mask_file_path is not None and Path(mask_file_path).exists():
        mask_volume = read_mha_xyz(mask_file_path).astype(np.int32)
        gt_raw = extract_slice(mask_volume, sag_axis, selected["idx"])
        gt = resize_lbl(group_sagittal_mask(gt_raw), IMG_SIZE)

    q = quality_flags(selected["pred"], selected["confidence"], min_fg=0.01, max_fg=0.45)

    return {
        "plane": "sagittal",
        "model_key": "sagittal_spider",
        "slice_index": int(selected["idx"]),
        "selection_score": float(selected["score"]),
        "image": selected["image"],
        "pred": selected["pred"],
        "prob": selected["prob"],
        "confidence": selected["confidence"],
        "gt": gt,
        "quality": q,
    }


if SAGITTAL_HOLDOUT_CASES.exists():
    sag_cases_df = pd.read_csv(SAGITTAL_HOLDOUT_CASES)
    print("Sagittal holdout rows:", len(sag_cases_df))
    display(sag_cases_df.head())
    print("Columns:", list(sag_cases_df.columns))
else:
    sag_cases_df = pd.DataFrame()
    print("No existe SAGITTAL_HOLDOUT_CASES. Se saltea demo sagital.")


In [ ]:
sag_rows = []
sag_metric_rows = []

if len(sag_cases_df) > 0:
    for i, (_, r) in enumerate(sag_cases_df.head(6).iterrows(), 1):
        img_path = resolve_spider_path(r, "image")
        mask_path = resolve_spider_path(r, "mask")

        if img_path is None:
            print("No se pudo resolver imagen para fila", i)
            continue

        result = infer_sagittal_spider(img_path, mask_path)

        if result["gt"] is not None:
            metrics = dice_by_class(
                result["pred"],
                result["gt"],
                num_classes=sag_num_classes,
                class_names=SAGITTAL_CLASS_NAMES,
            )
            metrics["example_id"] = i
            metrics["plane"] = "sagittal"
            metrics["case_id"] = candidate_case_id_from_row(r)
            sag_metric_rows.append(metrics)

        out_path = FIGURES_ROOT / f"E13_sagittal_example_{i:02d}.png"
        save_overlay_figure(
            result["image"],
            result["pred"],
            out_path,
            title=f"E13 sagittal example {i} | case={candidate_case_id_from_row(r)} | slice={result['slice_index']}",
            gt=result["gt"],
        )

        sag_rows.append({
            "example_id": i,
            "plane": "sagittal",
            "case_id": candidate_case_id_from_row(r),
            "slice_index": result["slice_index"],
            "selection_score": result["selection_score"],
            "image_path": str(img_path),
            "mask_path": str(mask_path) if mask_path else None,
            "figure_path": str(out_path),
            **result["quality"],
        })

sag_examples_df = pd.DataFrame(sag_rows)
sag_metrics_df = pd.concat(sag_metric_rows, ignore_index=True) if sag_metric_rows else pd.DataFrame()

sag_examples_df.to_csv(E13_ROOT / "E13_sagittal_examples_quality.csv", index=False)
sag_metrics_df.to_csv(E13_ROOT / "E13_sagittal_examples_metrics_by_class.csv", index=False)

display(sag_examples_df)
display(sag_metrics_df.head(20))


## 6. Reporte final E13

In [ ]:
def df_summary(df, name):
    return {
        "name": name,
        "rows": int(len(df)),
        "columns": list(df.columns) if len(df.columns) else [],
    }


report = {
    "notebook": "21_E13_multiplanar_inference_pipeline",
    "goal": "common inference pipeline for sagittal SPIDER and axial T2 models",
    "models": {
        "sagittal_spider": {
            "path": str(SAGITTAL_MODEL_PATH),
            "num_classes": sag_num_classes,
            "target_size": list(sag_target_size),
            "axis": sag_axis,
            "slice_strategy": sag_slice_strategy,
        },
        "axial_t2_alkafri": {
            "path": str(AXIAL_MODEL_PATH),
            "num_classes": 6,
            "target_size": [IMG_SIZE, IMG_SIZE],
        },
    },
    "outputs": {
        "axial_quality_csv": str(E13_ROOT / "E13_axial_examples_quality.csv"),
        "axial_metrics_csv": str(E13_ROOT / "E13_axial_examples_metrics_by_class.csv"),
        "sagittal_quality_csv": str(E13_ROOT / "E13_sagittal_examples_quality.csv"),
        "sagittal_metrics_csv": str(E13_ROOT / "E13_sagittal_examples_metrics_by_class.csv"),
    },
    "summaries": [
        df_summary(axial_examples_df, "axial_examples_quality"),
        df_summary(axial_metrics_df, "axial_examples_metrics_by_class"),
        df_summary(sag_examples_df, "sagittal_examples_quality"),
        df_summary(sag_metrics_df, "sagittal_examples_metrics_by_class"),
    ],
    "decision": "common_multiplanar_inference_pipeline_ready_for_agent_prototype",
}

report_path = E13_ROOT / "E13_multiplanar_pipeline_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

md = f"""# E13 - Pipeline común de inferencia multiplanar

## Objetivo

Este notebook implementa un pipeline común para ejecutar inferencia con los dos modelos finales del proyecto:

- Modelo sagital SPIDER consolidado en E12.
- Modelo axial T2 Al-Kafri/Sudirman entrenado en E10.

## Resultado

Se generaron funciones comunes para:

- cargar modelos,
- preprocesar imágenes,
- ejecutar inferencia 2D,
- generar máscaras,
- generar overlays,
- calcular métricas por clase cuando existe ground truth,
- calcular quality flags.

## Modelos usados

- Sagital: `{SAGITTAL_MODEL_PATH}`
- Axial T2: `{AXIAL_MODEL_PATH}`

## Salidas

- `{E13_ROOT / "E13_axial_examples_quality.csv"}`
- `{E13_ROOT / "E13_axial_examples_metrics_by_class.csv"}`
- `{E13_ROOT / "E13_sagittal_examples_quality.csv"}`
- `{E13_ROOT / "E13_sagittal_examples_metrics_by_class.csv"}`

## Decisión

El pipeline común queda listo para avanzar a E14, donde se implementará el prototipo de agente/orquestador IA.
"""

md_path = DOCS_ROOT / "E13_multiplanar_inference_pipeline_conclusion.md"
md_path.write_text(md, encoding="utf-8")

print("Reporte JSON:", report_path)
print("Reporte Markdown:", md_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
